In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


In [37]:
df = pd.read_csv("../data/encoded_dataset.csv")
df.head()

,age,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,capital-gains,capital-loss,incident_severity,incident_hour_of_the_day,number_of_vehicles_involved,...,insured_hobbies_hiking,insured_hobbies_kayaking,insured_hobbies_movies,insured_hobbies_paintball,insured_hobbies_polo,insured_hobbies_reading,insured_hobbies_skydiving,insured_hobbies_sleeping,insured_hobbies_video-games,insured_hobbies_yachting
0,48,1000,7.249862,0,466132,53300,0,2,5,1,...,False,False,False,False,False,False,False,True,False,False
1,42,2000,7.088592,5000000,468176,0,0,1,8,1,...,False,False,False,False,False,True,False,False,False,False
2,29,2000,7.254277,5000000,430632,35100,0,1,7,3,...,False,False,False,False,False,False,False,False,False,False
3,41,2000,7.256114,6000000,608117,48900,-62400,2,5,1,...,False,False,False,False,False,False,False,False,False,False
4,44,1000,7.368283,6000000,610706,66000,-46000,1,20,1,...,False,False,False,False,False,False,False,False,False,False


In [38]:
X = df.drop("fraud_reported", axis=1)
y = df["fraud_reported"]

In [39]:
print(X.shape)
print(y.shape)

(1000, 143)
(1000,)


In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [41]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(800, 143)
(200, 143)
(800,)
(200,)


verification before SMOTE operation

In [42]:
X_train.select_dtypes(include=['object']).columns

Index([], dtype='object')

In [43]:
X_train.isnull().sum().sum()

np.int64(554)

In [44]:
X_train = X_train.fillna(0)

In [45]:
y_train.unique()

array([0, 1])

In [46]:
print(X_train.dtypes.value_counts())
print(X_train.select_dtypes(include=['object']).columns)
print(y_train.unique())

bool       121
int64       19
float64      3
Name: count, dtype: int64
Index([], dtype='object')
[0 1]


In [47]:
print(X_train.shape)
print(y_train.shape)

(800, 143)
(800,)


**applying SMOTE only for training data**

In [48]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

checking class imbalance

In [49]:
y_train_smote.value_counts()

fraud_reported
0    602
1    602
Name: count, dtype: int64

In [ ]:
X_test_scaled = scaler.transform(X_test)

In [51]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)
print(X_train_smote.shape)

(1204, 143)
(200, 143)
(1204, 143)


verification(again)

In [52]:
print(X_train.isna().sum().sum())
print(X_test.isna().sum().sum())


0
149


In [53]:
print(X_train.columns[X_train.isna().any()])
print(X_test.columns[X_test.isna().any()])

Index([], dtype='object')
Index(['property_damage', 'police_report_available'], dtype='object')


In [54]:
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_test.median())

In [55]:
X_train['property_damage'].fillna(X_train['property_damage'].mode()[0], inplace=True)
X_test['property_damage'].fillna(X_train['property_damage'].mode()[0], inplace=True)

X_train['police_report_available'].fillna(X_train['police_report_available'].mode()[0], inplace=True)
X_test['police_report_available'].fillna(X_train['police_report_available'].mode()[0], inplace=True)

C:\Users\saiks\AppData\Local\Temp\ipykernel_11612\1253940731.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train['property_damage'].fillna(X_train['property_damage'].mode()[0], inplace=True)
C:\Users\saiks\AppData\Local\Temp\ipykernel_11612\1253940731.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting valu

In [56]:
print(X_train.isna().sum().sum())
print(X_test.isna().sum().sum())

0
0


**Handling Imbalanced Data**

**Logistic Regression**

**Approach A** : Train a model using the original data so you have a baseline performance to compare later.

In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Baseline model
baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_baseline))
print(confusion_matrix(y_test, y_pred_baseline))
print(classification_report(y_test, y_pred_baseline))

Accuracy: 0.745
[[149   2]
 [ 49   0]]
              precision    recall  f1-score   support

           0       0.75      0.99      0.85       151
           1       0.00      0.00      0.00        49

    accuracy                           0.74       200
   macro avg       0.38      0.49      0.43       200
weighted avg       0.57      0.74      0.64       200



c:\Users\saiks\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Approach B - SMOTE (only on Training Dataset)

In [58]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_smote.value_counts())

Before SMOTE: fraud_reported
0    602
1    198
Name: count, dtype: int64
After SMOTE: fraud_reported
0    602
1    602
Name: count, dtype: int64


In [59]:
smote_model = LogisticRegression(max_iter=1000)

smote_model.fit(X_train_smote, y_train_smote)

y_pred_smote = smote_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_smote))
print(confusion_matrix(y_test, y_pred_smote))
print(classification_report(y_test, y_pred_smote))

Accuracy: 0.565
[[90 61]
 [26 23]]
              precision    recall  f1-score   support

           0       0.78      0.60      0.67       151
           1       0.27      0.47      0.35        49

    accuracy                           0.56       200
   macro avg       0.52      0.53      0.51       200
weighted avg       0.65      0.56      0.59       200



c:\Users\saiks\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Approach C — Class Weight Balancing

In [60]:
weighted_model = LogisticRegression(class_weight='balanced', max_iter=1000)

weighted_model.fit(X_train, y_train)

y_pred_weighted = weighted_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_weighted))
print(confusion_matrix(y_test, y_pred_weighted))
print(classification_report(y_test, y_pred_weighted))

Accuracy: 0.515
[[78 73]
 [24 25]]
              precision    recall  f1-score   support

           0       0.76      0.52      0.62       151
           1       0.26      0.51      0.34        49

    accuracy                           0.52       200
   macro avg       0.51      0.51      0.48       200
weighted avg       0.64      0.52      0.55       200



c:\Users\saiks\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [61]:
X_train.shape[1]

143

**Now we try the 3 approaches for different models, to choose the best one**

In [62]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "Naive Bayes": GaussianNB()
}

Helper Functions

In [63]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    return {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0)
    }

In [64]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [65]:
results = []

for name, model in models.items():

    # ----- Approach A: Baseline -----
    res_base = evaluate_model(model, X_train, y_train, X_test, y_test)
    res_base["Model"] = name
    res_base["Approach"] = "Baseline"
    results.append(res_base)

    # ----- Approach B: SMOTE -----
    res_smote = evaluate_model(model, X_train_smote, y_train_smote, X_test, y_test)
    res_smote["Model"] = name
    res_smote["Approach"] = "SMOTE"
    results.append(res_smote)

    # ----- Approach C: Class Weight (only for supported models) -----
    if name in ["Decision Tree", "Random Forest", "SVM"]:
        if name == "Decision Tree":
            model_cw = DecisionTreeClassifier(class_weight='balanced', max_depth=10,min_samples_leaf=10,random_state=42)
        elif name == "Random Forest":
            model_cw = RandomForestClassifier(n_estimators=200, max_depth = 10, min_samples_leaf=5,class_weight='balanced', random_state=42)
        elif name == "SVM":
            model_cw = SVC(class_weight='balanced')

        res_cw = evaluate_model(model_cw, X_train, y_train, X_test, y_test)
        res_cw["Model"] = name
        res_cw["Approach"] = "Class Weight"
        results.append(res_cw)

        if name == "Decision Tree":
            dt_best = model_cw

In [66]:
import pandas as pd

df_results = pd.DataFrame(results)
df_results = df_results[["Model", "Approach", "Accuracy", "Precision", "Recall", "F1"]]

print(df_results)

            Model      Approach  Accuracy  Precision    Recall        F1
0   Decision Tree      Baseline     0.775   0.541667  0.530612  0.536082
1   Decision Tree         SMOTE     0.770   0.528302  0.571429  0.549020
2   Decision Tree  Class Weight     0.755   0.500000  0.714286  0.588235
3   Random Forest      Baseline     0.745   0.375000  0.061224  0.105263
4   Random Forest         SMOTE     0.765   0.538462  0.285714  0.373333
5   Random Forest  Class Weight     0.765   0.525000  0.428571  0.471910
6             KNN      Baseline     0.665   0.178571  0.102041  0.129870
7             KNN         SMOTE     0.495   0.234694  0.469388  0.312925
8             SVM      Baseline     0.755   0.000000  0.000000  0.000000
9             SVM         SMOTE     0.650   0.243902  0.204082  0.222222
10            SVM  Class Weight     0.650   0.243902  0.204082  0.222222
11    Naive Bayes      Baseline     0.750   0.454545  0.102041  0.166667
12    Naive Bayes         SMOTE     0.455   0.28571

Decision Tree - Class Weight Model seems to be the best fit model, so we'll proceed with that model

In [67]:
import joblib
joblib.dump(dt_best, "decision_tree_class_weight.pkl")

['decision_tree_class_weight.pkl']

In [ ]:



X_test = imputer.transform(X_test)



['imputer.pkl']

In [ ]:
joblib.dump(X_test_scaled, "X_test_scaled.pkl")
joblib.dump(y_test, "y_test.pkl")
joblib.dump(X.columns.tolist(), "feature_columns.pkl")



['feature_columns.pkl']

In [70]:
import numpy as np

# Check predictions on training data
train_pred = model.predict(X_train)

print("Fraud predictions in TRAIN:", np.sum(train_pred))
print("Actual fraud in TRAIN:", np.sum(y_train))

Fraud predictions in TRAIN: 493
Actual fraud in TRAIN: 198


c:\Users\saiks\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but GaussianNB was fitted with feature names
  warnings.warn(
